# 00 — Tải dữ liệu ảnh (Đề tài 9 · HVH)

**Đề tài:** HVH — Xây dựng ngữ liệu đơn ngữ chữ Hán chuyên ngành lịch sử VN (phong kiến)  
**Phạm vi:** 17 tác phẩm `HVH_246` → `HVH_262`, tổng **2180 trang**  
**Nguồn:** Thư viện số Nôm Foundation — https://lib.nomfoundation.org

Notebook này **chỉ làm khâu tải ảnh gốc** và tạo sẵn cấu trúc thư mục cho các bước sau
(OCR → tách câu → NER). Ảnh được lưu vào `data/raw_images/<mã tác phẩm>/`.

> URL ảnh của Nôm Foundation có dạng:  
> `https://lib.nomfoundation.org/site_media/nom/<docid>/jpeg/<docid>-NNN.jpg`  
> trong đó `docid` (vd `nlvnpf-1212`, `nlvnpf-0475-09`) được lấy tự động từ trang *volume*,
> còn `NNN` là số trang đệm 3 chữ số (`001`, `002`, ...).

## 1. Cấu trúc thư mục dự án

Notebook sẽ tự tạo bộ khung thư mục dưới đây (các bước sau dùng lại đúng cấu trúc này):

```
midterm/
├── notebooks/
│   ├── 00_download_data.ipynb   ← notebook hiện tại
│   ├── 01_preprocess.ipynb      (sẽ tạo sau: tiền xử lý ảnh)
│   ├── 02_ocr.ipynb             (OCR chữ Hán)
│   ├── 03_segment.ipynb         (tách câu / đoạn cú)
│   └── 04_ner.ipynb             (nhận dạng thực thể)
├── data/
│   ├── metadata/
│   │   └── de_tai_9.csv         ← bảng metadata 17 tác phẩm (notebook sinh ra)
│   ├── raw_images/              ← ẢNH GỐC tải về, mỗi tác phẩm 1 thư mục
│   │   ├── HVH_246/
│   │   │   ├── HVH_246_p001.jpg
│   │   │   └── ...
│   │   └── HVH_262/
│   └── preprocessed/            ← ảnh sau tiền xử lý (bước 01)
├── outputs/                     ← KẾT QUẢ NỘP, đúng format đề bài
│   ├── HVH_246/
│   │   ├── HVH_246_raw.txt      (OCR thô)
│   │   ├── HVH_246_seg.tsv      (tách câu)
│   │   └── HVH_246_ner.json     (NER)
│   └── ...  (tác phẩm nhiều quyển -> thư mục con HVH_258_01/, HVH_258_02/ ...)
├── gold_set/                    ← tập vàng để đánh giá/khảo sát model
└── src/                         ← code dùng chung (module .py)
```

## 2. Cài đặt & cấu hình đường dẫn

In [ ]:
# Nếu thiếu thư viện, bỏ comment dòng dưới để cài
# %pip install requests pandas tqdm pillow

import os, re, time, csv, json
from pathlib import Path
import requests
from tqdm.auto import tqdm

# --- Xác định thư mục gốc dự án (notebook nằm trong midterm/notebooks/) ---
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
print('Project root:', PROJECT_ROOT)

DATA_DIR      = PROJECT_ROOT / 'data'
META_DIR      = DATA_DIR / 'metadata'
RAW_IMG_DIR   = DATA_DIR / 'raw_images'
PREP_DIR      = DATA_DIR / 'preprocessed'
OUTPUTS_DIR   = PROJECT_ROOT / 'outputs'
GOLD_DIR      = PROJECT_ROOT / 'gold_set'
SRC_DIR       = PROJECT_ROOT / 'src'

# --- Tạo bộ khung thư mục ---
for d in [META_DIR, RAW_IMG_DIR, PREP_DIR, OUTPUTS_DIR, GOLD_DIR, SRC_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Đã tạo/khởi tạo cấu trúc thư mục xong.')

## 3. Metadata 17 tác phẩm của Đề tài 9

Bảng dưới lấy từ file *DS Đề tài KHDL.xlsx*. Cột `volume_url` là trang *volume* trên
Nôm Foundation để tự động dò `docid` và tải ảnh.

⚠️ **Một số tác phẩm trong Excel không có link volume trực tiếp** (chỉ có link *subject listing*
hoặc chỉ ghi tên) — được đánh dấu `None`. Với các mục này, bạn cần **tự mở nomfoundation,
tìm đúng trang volume rồi điền URL vào** (xem phần *MANUAL_URLS* bên dưới). Mình đã điền sẵn
vài URL **phỏng đoán** (theo dãy volume id liền kề) và **đánh dấu cần kiểm tra**.

In [ ]:
# (code, tên tác phẩm, tên gốc Hán, volume_url | None, số trang theo Excel)
WORKS = [
    ("HVH_246", "Thăng Hàm Nhật Kí", "陞銜日記", None, 394),
    ("HVH_247", "Thăng Thưởng Nhật Kí", "陞賞日記", None, 747),
    ("HVH_248", "Thăng Thưởng Nhật Kí Văn Giai", "陞賞日記文階", "https://lib.nomfoundation.org/collection/1/volume/405/", 81),
    ("HVH_249", "Thăng Thưởng Nhật Kí Võ Giai", "陞賞日記武階", None, 44),
    ("HVH_250", "Thăng Thưởng Phẩm Hàm Nhật Kí", "陞賞品銜日記", None, 220),
    ("HVH_251", "Thiên Nam Địa Thế Khai Chính Địa Lý Quốc Ngữ", "天南地勢開正地里國語", "https://lib.nomfoundation.org/collection/1/volume/1169/", 39),
    ("HVH_252", "Thiên Nam Nhân Vật", "天南人物", "https://lib.nomfoundation.org/collection/1/volume/1171/", 74),
    ("HVH_253", "Thiên Nam Tứ Tự Kinh", "天南四字經", None, 35),
    ("HVH_254", "Thiệu Trị Đinh Mùi Khoa Hội Thí", "紹治丁未科會試", "https://lib.nomfoundation.org/collection/1/volume/1179/", 37),
    ("HVH_255", "Thiệu Trị Nhị Niên Nhâm Dần Khoa Hương Thí Văn", "紹治二年壬寅科鄕試文選", "https://lib.nomfoundation.org/collection/1/volume/1180/", 41),
    ("HVH_256", "Thiệu Trị Tam Niên Quý Mão Khoa Đình Thí Hội Thí", "紹治三年癸卯科庭試會試", "https://lib.nomfoundation.org/collection/1/volume/1181/", 29),
    ("HVH_257", "Thuỵ Ứng Gia Phả", "瑞應家譜", "https://lib.nomfoundation.org/collection/1/volume/429/", 9),
    ("HVH_258", "Thư Kinh Đại Toàn (Q.01-02)", "書經大全", "https://lib.nomfoundation.org/collection/1/volume/627/", 99),
    ("HVH_259", "Thư Kinh Đại Toàn (Q.05-06)", "書經大全", None, 108),
    ("HVH_260", "Thư Kinh Đại Toàn (Q.09-10)", "書經大全", "https://lib.nomfoundation.org/collection/1/volume/629/", 118),
    ("HVH_261", "Thư Kinh Đại Toàn (Q.Thủ)", "書經大全", "https://lib.nomfoundation.org/collection/1/volume/626/", 90),
    ("HVH_262", "Thừa Sao Duệ Hiệu", "承抄睿號", None, 15),
]

# URL do bạn tự tìm/điền cho các tác phẩm chưa có link volume trực tiếp.
# (*) = URL PHỎNG ĐOÁN theo dãy volume id liền kề -> HÃY MỞ TRÌNH DUYỆT KIỂM TRA trước khi tải!
MANUAL_URLS = {
    # 'HVH_246': 'https://lib.nomfoundation.org/collection/1/volume/XXXX/',  # Thăng Hàm Nhật Kí — cần tìm
    # 'HVH_247': 'https://lib.nomfoundation.org/collection/1/volume/XXXX/',  # Thăng Thưởng Nhật Kí — cần tìm
    # 'HVH_250': 'https://lib.nomfoundation.org/collection/1/volume/XXXX/',  # Thăng Thưởng Phẩm Hàm Nhật Kí — cần tìm
    # 'HVH_253': 'https://lib.nomfoundation.org/collection/1/volume/XXXX/',  # Thiên Nam Tứ Tự Kinh — cần tìm
    'HVH_259': 'https://lib.nomfoundation.org/collection/1/volume/628/',    # (*) Q.05-06 giữa 627 và 629 — KIỂM TRA
    # 'HVH_262': 'https://lib.nomfoundation.org/collection/1/volume/XXXX/',  # Thừa Sao Duệ Hiệu — cần tìm
}

# Gộp MANUAL_URLS đè lên WORKS
WORKS = [(c, n, g, MANUAL_URLS.get(c, u), p) for (c, n, g, u, p) in WORKS]

n_have = sum(1 for w in WORKS if w[3])
print(f'Có {n_have}/{len(WORKS)} tác phẩm đã có URL volume để tải tự động.')
print('Chưa có URL (cần điền MANUAL_URLS):', [w[0] for w in WORKS if not w[3]])

### Lưu metadata ra `data/metadata/de_tai_9.csv`

In [ ]:
meta_path = META_DIR / 'de_tai_9.csv'
with open(meta_path, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['code', 'ten_su_lieu', 'ten_goc_han', 'volume_url', 'so_trang'])
    for row in WORKS:
        w.writerow(row)
print('Đã lưu metadata:', meta_path)

import pandas as pd
df = pd.read_csv(meta_path)
df

## 4. Hàm tải ảnh

- `get_docid(volume_url)`: mở trang volume, dò `docid` từ đường dẫn ảnh đầu tiên.
- `download_work(...)`: tải tuần tự `-001.jpg`, `-002.jpg`, ... **cho tới khi gặp trang 404**
  (đồng thời dừng nếu vượt quá số trang kỳ vọng + biên an toàn).
- **Hỗ trợ resume:** file đã tải thì bỏ qua, chạy lại notebook không tải trùng.
- Có delay lịch sự giữa các request để không làm phiền máy chủ thư viện.

In [ ]:
BASE = 'https://lib.nomfoundation.org'
IMG_TMPL = BASE + '/site_media/nom/{docid}/jpeg/{docid}-{n:03d}.jpg'
HEADERS = {'User-Agent': 'Mozilla/5.0 (research; midterm NLP corpus builder)'}

session = requests.Session()
session.headers.update(HEADERS)

DOCID_RE = re.compile(r'/site_media/nom/([A-Za-z0-9_\-]+)/jpeg/')

def get_docid(volume_url, timeout=30):
    """Lấy docid (vd nlvnpf-1212) từ trang volume."""
    r = session.get(volume_url, timeout=timeout)
    r.raise_for_status()
    m = DOCID_RE.search(r.text)
    if not m:
        raise RuntimeError(f'Không tìm thấy docid trong trang: {volume_url}')
    return m.group(1)

def download_image(url, dest, timeout=60):
    """Tải 1 ảnh. Trả về True nếu tải/đã có, False nếu 404 (hết trang)."""
    if dest.exists() and dest.stat().st_size > 0:
        return True
    r = session.get(url, timeout=timeout)
    if r.status_code == 404:
        return False
    r.raise_for_status()
    # trang không tồn tại đôi khi trả HTML thay vì ảnh
    ctype = r.headers.get('Content-Type', '')
    if 'image' not in ctype:
        return False
    dest.write_bytes(r.content)
    return True

In [ ]:
# Lưu ý: số trang trong Excel đôi khi ĐẾM THIẾU so với ảnh thật trên site,
# nên KHÔNG chặn cứng theo expected_pages — chỉ dừng khi gặp trang 404.
ABS_MAX_PAGES = 2000  # chặn an toàn tuyệt đối, phòng vòng lặp vô hạn

def download_work(code, name, volume_url, expected_pages=None, delay=0.4):
    """Tải toàn bộ ảnh của 1 tác phẩm vào data/raw_images/<code>/."""
    if not volume_url:
        print(f'[SKIP] {code} — chưa có volume_url (điền vào MANUAL_URLS).')
        return {'code': code, 'status': 'no_url', 'downloaded': 0}

    out_dir = RAW_IMG_DIR / code
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        docid = get_docid(volume_url)
    except Exception as e:
        print(f'[ERR ] {code} — không lấy được docid: {e}')
        return {'code': code, 'status': 'docid_error', 'downloaded': 0}

    hard_limit = ABS_MAX_PAGES
    n_ok = 0
    pbar = tqdm(range(1, hard_limit + 1), desc=f'{code} [{docid}]', leave=True)
    for n in pbar:
        url = IMG_TMPL.format(docid=docid, n=n)
        dest = out_dir / f'{code}_p{n:03d}.jpg'
        try:
            ok = download_image(url, dest)
        except Exception as e:
            print(f'  ! lỗi trang {n} của {code}: {e} — thử lại sau 2s')
            time.sleep(2)
            try:
                ok = download_image(url, dest)
            except Exception as e2:
                print(f'  !! bỏ qua trang {n}: {e2}')
                continue
        if not ok:
            pbar.set_postfix_str(f'hết trang ở {n-1}')
            break
        n_ok += 1
        time.sleep(delay)

    status = 'ok'
    if expected_pages and n_ok < expected_pages:
        status = f'thiếu ({n_ok}/{expected_pages})'
    print(f'[DONE] {code}: tải {n_ok} trang (kỳ vọng {expected_pages}) -> {out_dir}')
    return {'code': code, 'docid': docid, 'status': status, 'downloaded': n_ok}

## 5. Chạy thử trên 1 tác phẩm nhỏ trước

Nên test 1 tác phẩm ít trang (vd `HVH_257` Thuỵ Ứng Gia Phả — 9 trang) để chắc pipeline chạy đúng,
kiểm tra ảnh mở được, rồi mới tải toàn bộ.

In [ ]:
test = next(w for w in WORKS if w[0] == 'HVH_257')
code_, name_, goc_, url_, pages_ = test
res = download_work(code_, name_, url_, expected_pages=pages_)
res

In [ ]:
# Xem thử ảnh trang đầu vừa tải
from PIL import Image
import matplotlib.pyplot as plt
p = RAW_IMG_DIR / 'HVH_257' / 'HVH_257_p001.jpg'
if p.exists():
    img = Image.open(p)
    print('Kích thước ảnh:', img.size)
    plt.figure(figsize=(6, 9)); plt.imshow(img); plt.axis('off'); plt.show()
else:
    print('Chưa có ảnh — kiểm tra lại bước tải.')

## 6. Tải toàn bộ Đề tài 9

Chạy cell dưới để tải hết 17 tác phẩm (những mục chưa có URL sẽ bị bỏ qua với cảnh báo).
Có thể chạy lại nhiều lần — file đã tải sẽ được **bỏ qua (resume)**.

In [ ]:
summary = []
for (code_, name_, goc_, url_, pages_) in WORKS:
    summary.append(download_work(code_, name_, url_, expected_pages=pages_))

import pandas as pd
sum_df = pd.DataFrame(summary)
sum_df.to_csv(META_DIR / 'download_summary.csv', index=False, encoding='utf-8-sig')
sum_df

## 7. Kiểm tra tổng số ảnh đã tải

In [ ]:
total = 0
print(f'{"Mã":<10}{"Số ảnh":>8}   Thư mục')
print('-' * 50)
for (code_, *_rest) in WORKS:
    d = RAW_IMG_DIR / code_
    n = len(list(d.glob(f'{code_}_p*.jpg'))) if d.exists() else 0
    total += n
    print(f'{code_:<10}{n:>8}   {d}')
print('-' * 50)
print(f'{"TỔNG":<10}{total:>8}  (kỳ vọng ~2180)')

## 8. Bước tiếp theo

- **Với các tác phẩm còn `no_url`** (HVH_246, 247, 250, 253, 262, ...): mở
  https://lib.nomfoundation.org, tìm đúng trang *volume*, điền URL vào `MANUAL_URLS`
  ở mục 3 rồi chạy lại mục 6.
- Ảnh đã nằm gọn trong `data/raw_images/<mã>/` → sang notebook **`01_preprocess.ipynb`**
  (khử nhiễu, deskew, tách cột) rồi **`02_ocr.ipynb`**.
- Kết quả cuối cùng sẽ ghi vào `outputs/<mã>/` đúng format `_raw.txt`, `_seg.tsv`, `_ner.json`.